# Ditto TalkingHead - Frozen Head Inference

End-to-end notebook for the **frozen-head** fork of
[Ditto](https://github.com/antgroup/ditto-talkinghead):

> `https://github.com/MoshiHead/ditto-talkinghead-original-freeze-head`

The head stays **exactly** in the pose of the source image - zero rigid motion
(no nodding, turning, tilting, drifting or scaling). Everything else is
untouched: lips, jaw, eye blinks, gaze and all other audio-driven expression.

**Inputs** (shipped with the repo):

| | |
|---|---|
| Audio | `example/audio.wav` |
| Image | `example/image.jpeg` |

**Output:** `tmp/result_frozen_head.mp4`

---

### Before you run

Set the runtime to a GPU: **Runtime -> Change runtime type -> T4 GPU** (or any
NVIDIA GPU). Then run every cell top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi

import sys, platform
print("\npython :", sys.version.split()[0], "|", platform.platform())

## 2. Clone the frozen-head repository

If you are already running this notebook from inside a cloned checkout, the
cell detects it and skips the clone.

In [ ]:
import os, subprocess, sys

REPO_URL  = "https://github.com/MoshiHead/ditto-talkinghead-original-freeze-head.git"
REPO_NAME = "ditto-talkinghead-original-freeze-head"

# where to work: /content on Colab, cwd otherwise
BASE_DIR = "/content" if os.path.isdir("/content") else os.getcwd()


def _is_ditto_root(p):
    return os.path.isfile(os.path.join(p, "inference.py")) and \
           os.path.isdir(os.path.join(p, "core"))


if _is_ditto_root(os.getcwd()):
    REPO_DIR = os.getcwd()
    print("Already inside a Ditto checkout, skipping clone.")
else:
    REPO_DIR = os.path.join(BASE_DIR, REPO_NAME)
    if _is_ditto_root(REPO_DIR):
        print(f"{REPO_DIR} already exists, pulling latest ...")
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("\nREPO_DIR =", REPO_DIR)
print("contents :", sorted(os.listdir(REPO_DIR)))
assert _is_ditto_root(REPO_DIR), "clone failed - inference.py / core/ not found"

## 3. Install dependencies

### 3a. System libraries

`mediapipe` (the 478-point face mesh) links against EGL/OpenGL, and the Colab
image does not ship those. Without them, importing it dies with:

```
OSError: libEGL.so.1: cannot open shared object file: No such file or directory
```

This cell installs them. **Run it before anything imports `mediapipe`** - it
is the first thing `StreamSDK` pulls in.

In [ ]:
%%bash
# EGL / OpenGL / GLib runtime needed by mediapipe (and by non-headless OpenCV).
# Package names differ slightly across Ubuntu releases, so try the variants and
# keep whatever installs.
apt-get -qq update 2>&1 | tail -1

for pkg in libegl1 libegl1-mesa libgl1 libgl1-mesa-glx libglx-mesa0 \
           libgles2 libgles2-mesa libopengl0 libglib2.0-0 libglib2.0-0t64 \
           libsm6 libxext6 libxrender1 ffmpeg; do
    apt-get -qq install -y --no-install-recommends "$pkg" >/dev/null 2>&1 \
        && echo "installed  $pkg" \
        || echo "skipped    $pkg (not available on this release)"
done

echo
echo "--- libEGL / libGL now present? ---"
ldconfig -p | grep -E "libEGL\.so\.1|libGL\.so\.1|libGLdispatch" || echo "STILL MISSING"

### 3b. Python packages

Colab already ships a CUDA build of PyTorch, so we only add what Ditto needs on
top of it:

* `librosa` - audio loading / resampling
* `onnxruntime-gpu` - auxiliary models (face detection, 106/203 landmarks, HuBERT)
* `mediapipe` - the 478-point face mesh
* `imageio` + `imageio-ffmpeg` - video writing
* `filetype`, `scikit-image`, `cython`, `colored`, `einops`, `tqdm` - misc

> We use the **PyTorch** checkpoints rather than TensorRT. That avoids pinning
> `tensorrt==8.6.1` and the `Ampere_Plus` engine compatibility problem, so this
> notebook runs on any CUDA GPU (T4, L4, A100, ...).

In [ ]:
%pip install -q librosa tqdm filetype imageio imageio-ffmpeg scikit-image cython colored einops "huggingface_hub>=0.23"

try:
    import mediapipe
    print("mediapipe already present:", mediapipe.__version__)
except ImportError:
    get_ipython().run_line_magic("pip", "install -q mediapipe")

print("\ndone")

### 3c. Match onnxruntime to the CUDA version torch was built against

The current PyPI `onnxruntime-gpu` wheel is built for **CUDA 13**, while Colab's
PyTorch ships **CUDA 12** libraries. Mixing them makes the CUDA execution
provider fail to load:

```
[E:onnxruntime:...] Failed to load library libonnxruntime_providers_cuda.so
with error: libcublasLt.so.13: cannot open shared object file
```

That one is *not* fatal - onnxruntime silently falls back to the CPU provider
and inference still completes - but HuBERT feature extraction then runs on CPU,
which is the slowest part of the pipeline. This cell installs a CUDA-12 build
instead and preloads the CUDA libraries that torch bundles, so the GPU provider
actually binds.

Set `FIX_ORT_CUDA = False` to skip it and just run the aux models on CPU.

In [ ]:
FIX_ORT_CUDA = True          # False -> leave onnxruntime alone, aux models on CPU
ORT_CUDA12_PIN = "onnxruntime-gpu==1.22.0"   # last series built against CUDA 12

import ctypes, glob, os, subprocess, sys, importlib.util

import torch
torch_cuda = (torch.version.cuda or "")
print("torch:", torch.__version__, "| torch CUDA:", torch_cuda or "cpu-only")


# RTLD_GLOBAL-load the CUDA libs pip installed next to torch.
#
# onnxruntime's CUDA provider dlopen()s libcublas / libcublasLt / libcudnn by
# soname. On Colab those live in site-packages/nvidia/*/lib, which is not on the
# loader path, so we load them into the process up front and also add the
# directories to LD_LIBRARY_PATH for any child process.
def preload_torch_cuda_libs():
    roots = []
    for sp in sys.path:
        d = os.path.join(sp, "nvidia")
        if os.path.isdir(d):
            roots.append(d)
    libdirs, loaded = set(), []
    wanted = ("libcudart", "libcublas", "libcublasLt", "libcudnn",
              "libcufft", "libcurand", "libcusparse", "libnvrtc")
    for root in roots:
        for so in sorted(glob.glob(os.path.join(root, "*", "lib", "*.so*"))):
            base = os.path.basename(so)
            if not base.startswith(wanted):
                continue
            libdirs.add(os.path.dirname(so))
            try:
                ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
                loaded.append(base)
            except OSError:
                pass
    if libdirs:
        os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(
            sorted(libdirs) + [os.environ.get("LD_LIBRARY_PATH", "")]
        ).strip(os.pathsep)
    return sorted(set(loaded)), sorted(libdirs)


# Ask the loader itself which shared objects the CUDA provider cannot resolve,
# rather than guessing from the onnxruntime version. Returns [] when the wheel
# is CPU-only (nothing to fix) or when everything resolves.
def cuda_ep_unresolved():
    if importlib.util.find_spec("onnxruntime") is None:
        return None
    import onnxruntime
    so = os.path.join(os.path.dirname(onnxruntime.__file__), "capi",
                      "libonnxruntime_providers_cuda.so")
    if not os.path.isfile(so):
        return []                      # CPU-only wheel
    out = subprocess.run(["ldd", so], capture_output=True, text=True).stdout
    return [l.split("=>")[0].strip() for l in out.splitlines() if "not found" in l]


# 1. make torch's bundled CUDA 12 libs visible first, so the check below only
#    reports sonames that are genuinely from the wrong CUDA major version
libs, dirs = preload_torch_cuda_libs()
print(f"\npreloaded {len(libs)} CUDA libs from {len(dirs)} dir(s)")
for d in dirs:
    print("  ", d)

# 2. now see what is still missing
if FIX_ORT_CUDA:
    unresolved = cuda_ep_unresolved()
    if unresolved is None:
        print("\nonnxruntime not installed -> installing", ORT_CUDA12_PIN)
        unresolved = ["<not installed>"]
    elif unresolved == []:
        import onnxruntime as _o
        print(f"\nonnxruntime {_o.__version__}: CUDA provider deps all resolve, "
              "nothing to fix")
    else:
        import onnxruntime as _o
        print(f"\nonnxruntime {_o.__version__}: CUDA provider cannot resolve "
              f"{unresolved}")
        print("-> wrong CUDA major version for this torch build")

    if unresolved:
        print(f"\ninstalling {ORT_CUDA12_PIN} ...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "--force-reinstall", "--no-deps", ORT_CUDA12_PIN],
            check=True,
        )
        # drop the already-imported module so the new build is picked up
        for m in [k for k in list(sys.modules) if k.startswith("onnxruntime")]:
            del sys.modules[m]
        still = cuda_ep_unresolved()
        if still:
            print(f"still unresolved: {still}")
            print("-> Runtime > Restart session, then rerun from cell 3a")
        else:
            print("CUDA provider deps resolve now")
else:
    print("\nFIX_ORT_CUDA = False - aux models will run on CPU")

In [ ]:
# ---- import check, EGL check, onnxruntime provider check, ffmpeg -----------
import importlib, shutil, os

mods = ["numpy", "scipy", "cv2", "torch", "librosa", "imageio", "filetype",
        "skimage", "einops", "onnxruntime", "mediapipe", "tqdm"]
missing = []
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"{m:14s} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        missing.append((m, repr(e)))
        print(f"{m:14s} FAILED: {e}")

import torch
print("\ntorch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

# --- mediapipe is the one that needs libEGL; prove it really imports ---
try:
    from mediapipe.tasks.python import vision, BaseOptions
    print("mediapipe vision tasks: OK (libEGL resolved)")
except OSError as e:
    print(f"\n!! mediapipe still broken: {e}")
    print("   -> rerun cell 3a, then Runtime > Restart session and rerun from 3a")
except Exception as e:
    print(f"\n!! mediapipe import problem: {e!r}")

# --- which onnxruntime providers actually bind? ---
try:
    import onnxruntime as ort
    avail = ort.get_available_providers()
    print("\nonnxruntime", ort.__version__, "providers:", avail)
    if "CUDAExecutionProvider" not in avail:
        print("   note: CPU-only onnxruntime - aux models + HuBERT run on CPU "
              "(slower, but correct)")
except Exception as e:
    print("\n!! onnxruntime unavailable:", repr(e))

# inference.py muxes the audio with a plain `ffmpeg` call -> ensure it is on PATH
if shutil.which("ffmpeg") is None:
    import imageio_ffmpeg
    exe = imageio_ffmpeg.get_ffmpeg_exe()
    try:
        os.makedirs("/usr/local/bin", exist_ok=True)
        shutil.copy(exe, "/usr/local/bin/ffmpeg")
    except Exception:
        os.environ["PATH"] = os.path.dirname(exe) + os.pathsep + os.environ["PATH"]
    print("\nffmpeg wired up from imageio-ffmpeg:", shutil.which("ffmpeg"))
else:
    print("\nffmpeg:", shutil.which("ffmpeg"))

if missing:
    print("\n!! missing modules:", [m for m, _ in missing])
else:
    print("\nall imports OK")

## 4. Download the checkpoints

From [digital-avatar/ditto-talkinghead](https://huggingface.co/digital-avatar/ditto-talkinghead).
We pull only what the PyTorch pipeline needs (`ditto_cfg/` + `ditto_pytorch/`,
about 1 GB) instead of the full repo, which also holds the ONNX and TensorRT
variants.

Set `BACKEND = "trt"` below if you specifically want the TensorRT engines
(you then also need `pip install tensorrt==8.6.1` and an Ampere+ GPU).

In [ ]:
BACKEND = "pytorch"   # "pytorch" (recommended) | "trt"

import os
from huggingface_hub import snapshot_download

HF_REPO  = "digital-avatar/ditto-talkinghead"
CKPT_DIR = os.path.join(os.getcwd(), "checkpoints")

if BACKEND == "pytorch":
    patterns  = ["ditto_cfg/*", "ditto_pytorch/*"]
    DATA_ROOT = os.path.join(CKPT_DIR, "ditto_pytorch")
    CFG_PKL   = os.path.join(CKPT_DIR, "ditto_cfg", "v0.4_hubert_cfg_pytorch.pkl")
elif BACKEND == "trt":
    patterns  = ["ditto_cfg/*", "ditto_trt_Ampere_Plus/*", "ditto_onnx/*"]
    DATA_ROOT = os.path.join(CKPT_DIR, "ditto_trt_Ampere_Plus")
    CFG_PKL   = os.path.join(CKPT_DIR, "ditto_cfg", "v0.4_hubert_cfg_trt.pkl")
else:
    raise ValueError(BACKEND)

snapshot_download(
    repo_id=HF_REPO,
    local_dir=CKPT_DIR,
    allow_patterns=patterns,
    max_workers=8,
)

print("\nDATA_ROOT =", DATA_ROOT)
print("CFG_PKL   =", CFG_PKL)

In [ ]:
# ---- verify the checkpoint tree --------------------------------------------
import os

def tree(root, prefix=""):
    if not os.path.isdir(root):
        print(prefix + "!! MISSING: " + root)
        return
    entries = [e for e in sorted(os.listdir(root)) if not e.startswith(".")]
    for i, name in enumerate(entries):
        p = os.path.join(root, name)
        last = i == len(entries) - 1
        branch = "`-- " if last else "|-- "
        if os.path.isdir(p):
            print(prefix + branch + name + "/")
            tree(p, prefix + ("    " if last else "|   "))
        else:
            print(f"{prefix}{branch}{name}  ({os.path.getsize(p) / 1e6:.1f} MB)")

tree(CKPT_DIR)

assert os.path.isfile(CFG_PKL), f"config pkl not found: {CFG_PKL}"
assert os.path.isdir(DATA_ROOT), f"data root not found: {DATA_ROOT}"
print("\ncheckpoints OK")

## 5. Verify the head-freeze patch

The freeze lives in `core/atomic_components/motion_stitch.py`. The motion the
diffusion model predicts is a 265-dim vector that splits into:

| part | keys | frozen? |
|---|---|---|
| rigid head pose | `pitch`, `yaw`, `roll`, `t`, `scale` | **yes - taken from the source** |
| expression | `exp` (lips, jaw, eyes, brows, ...) | no - fully audio-driven |

Three things happen when `freeze_head=True` (the default):

1. the rigid keys are stripped from `use_d_keys`, so they are never read from
   the driving sequence;
2. the pose control kwargs (`delta_pitch` / `delta_yaw` / `delta_roll`,
   `alpha_pitch` / `alpha_yaw` / `alpha_roll`) are dropped, so they cannot
   re-introduce head motion;
3. as the **last** step before the keypoint transform - after relative-motion
   mixing, ctrl and fade - the rigid keys are hard-reset to the source values.

The cell below asserts the patch is present and that it really produces zero
rigid motion. It stubs out the stitching network, so no weights are needed and
it runs in a second.

In [ ]:
import inspect
import numpy as np

import core.atomic_components.motion_stitch as ms_mod

# --- the patch is present ---
assert hasattr(ms_mod, "HEAD_POSE_KEYS"), "freeze patch missing from motion_stitch.py"
assert "freeze_head" in inspect.signature(ms_mod.MotionStitch.setup).parameters
print("HEAD_POSE_KEYS :", ms_mod.HEAD_POSE_KEYS)
print("HEAD_CTRL_KEYS :", ms_mod.HEAD_CTRL_KEYS)
print("default        : freeze_head =",
      inspect.signature(ms_mod.MotionStitch.setup).parameters["freeze_head"].default)


# --- numeric proof, with the stitch network stubbed out ---
class _IdentityStitch:
    def __init__(self, **kw):
        pass

    def __call__(self, x_s, x_d):
        return x_d


_real = ms_mod.StitchNetwork
ms_mod.StitchNetwork = _IdentityStitch
try:
    def make_info(seed, exp_seed=None):
        r = np.random.RandomState(seed)
        e = np.random.RandomState(seed if exp_seed is None else exp_seed)
        return {
            "scale": r.rand(1, 1).astype(np.float32) + 1,
            "pitch": r.rand(1, 66).astype(np.float32),
            "yaw":   r.rand(1, 66).astype(np.float32),
            "roll":  r.rand(1, 66).astype(np.float32),
            "t":     r.rand(1, 3).astype(np.float32),
            "exp":   e.rand(1, 63).astype(np.float32) * 0.1,
        }

    x_s_info = dict(make_info(1),
                    kp=np.random.RandomState(7).rand(1, 63).astype(np.float32))

    def drive(freeze, n=6, ctrl=None, fixed_exp=False):
        st = ms_mod.MotionStitch({})
        st.N_d = -1
        st.setup(N_d=n, is_image_flag=True, x_s_info=x_s_info,
                 freeze_head=freeze, drive_eye=True, delta_eye_open_n=-1)
        out = []
        for i in range(n):
            d = make_info(100 + i, exp_seed=999 if fixed_exp else None)
            _, x_d = st(dict(x_s_info), d, **(ctrl or {}))
            out.append(x_d[0].copy())
        return np.stack(out)            # (n, 21, 3)

    # Same expression every frame but a different driving head pose:
    # frozen -> the output must be bit-identical; original -> it must move.
    frozen   = drive(True,  fixed_exp=True)
    original = drive(False, fixed_exp=True)
    rigid_frozen   = np.abs(frozen   - frozen[0]).max()
    rigid_original = np.abs(original - original[0]).max()

    # pose ctrl kwargs must not leak through
    with_ctrl = drive(True, ctrl={"delta_yaw": 30.0, "delta_pitch": -20.0,
                                  "delta_roll": 10.0, "alpha_yaw": 4.0,
                                  "alpha_pitch": 4.0, "alpha_roll": 4.0})
    baseline  = drive(True)
    ctrl_leak = np.abs(with_ctrl - baseline).max()

    # expression must still be driven
    lip = [6, 12, 14, 17, 19, 20]
    eye = [11, 13, 15, 16, 18]
    lip_motion = baseline[:, lip].std(axis=0).max()
    eye_motion = baseline[:, eye].std(axis=0).max()

    print(f"\nrigid motion, frozen      : {rigid_frozen:.3e}   (must be 0)")
    print(f"rigid motion, original    : {rigid_original:.3e}   (sanity: > 0)")
    print(f"pose-ctrl leakage         : {ctrl_leak:.3e}   (must be 0)")
    print(f"lip motion still present  : {lip_motion:.3e}")
    print(f"eye motion still present  : {eye_motion:.3e}")

    assert rigid_frozen == 0.0,   "head still moves while frozen!"
    assert rigid_original > 1e-3, "baseline sanity check failed"
    assert ctrl_leak == 0.0,      "pose ctrl kwargs leaked through!"
    assert lip_motion > 1e-4,     "lips are not moving!"
    assert eye_motion > 1e-6,     "eyes are not moving!"
    print("\nHEAD FREEZE VERIFIED - rigid pose constant, expression intact")
finally:
    ms_mod.StitchNetwork = _real

## 6. Inputs

In [ ]:
import os, math
import librosa
import cv2
import matplotlib.pyplot as plt

AUDIO_PATH  = os.path.join(os.getcwd(), "example", "audio.wav")
SOURCE_PATH = os.path.join(os.getcwd(), "example", "image.jpeg")
OUTPUT_PATH = os.path.join(os.getcwd(), "tmp", "result_frozen_head.mp4")
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

assert os.path.isfile(AUDIO_PATH), AUDIO_PATH
assert os.path.isfile(SOURCE_PATH), SOURCE_PATH

wav, sr = librosa.core.load(AUDIO_PATH, sr=16000)
dur = len(wav) / sr
n_frames = math.ceil(dur * 25)
img = cv2.cvtColor(cv2.imread(SOURCE_PATH), cv2.COLOR_BGR2RGB)

print(f"audio : {AUDIO_PATH}")
print(f"        {dur:.2f}s @ 16kHz  ->  {n_frames} frames @ 25fps")
print(f"image : {SOURCE_PATH}  {img.shape[1]}x{img.shape[0]}")
print(f"output: {OUTPUT_PATH}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(img); ax[0].set_title("source image"); ax[0].axis("off")
ax[1].plot(wav, lw=0.4); ax[1].set_title(f"audio waveform ({dur:.2f}s)")
ax[1].set_xlim(0, len(wav)); ax[1].set_xticks([])
plt.tight_layout(); plt.show()

## 7. Build the SDK and run inference

`freeze_head=True` is the default, so a plain call already produces a frozen
head; it is passed explicitly here so the intent is visible.

Building the SDK takes a minute or two (all networks are loaded). Generation
then runs at a few frames per second on a T4 - the `dit` progress bar is the
diffusion sampler, `writer` is the video encoder.

In [ ]:
import sys

# --- preflight: the two native deps that typically fail on a fresh Colab ----
try:
    from mediapipe.tasks.python import vision, BaseOptions
except OSError as e:
    raise SystemExit(
        f"mediapipe cannot load its native library ({e}).\n"
        "Run cell 3a (system libraries), then Runtime > Restart session, "
        "then rerun from 3a."
    )

import onnxruntime as ort
if "CUDAExecutionProvider" not in ort.get_available_providers():
    print("note: onnxruntime has no CUDA provider -> aux models and HuBERT "
          "run on CPU. Inference is correct, just slower.\n")

# make sure the repo modules are (re)loaded from REPO_DIR
for m in [k for k in list(sys.modules)
          if k == "inference" or k.startswith(("core", "stream_pipeline"))]:
    del sys.modules[m]

from stream_pipeline_offline import StreamSDK

SDK = StreamSDK(CFG_PKL, DATA_ROOT)
print("\nSDK ready")

In [ ]:
import time
from inference import run, seed_everything

seed_everything(1024)          # reproducible sampling; comment out for variety

t0 = time.time()
run(
    SDK,
    AUDIO_PATH,
    SOURCE_PATH,
    OUTPUT_PATH,
    freeze_head=True,          # <-- head stays exactly as in the source image
)
print(f"\nfinished in {time.time() - t0:.1f}s")

In [ ]:
import os, subprocess, json

assert os.path.isfile(OUTPUT_PATH), "no output produced"
print(f"{OUTPUT_PATH}  ({os.path.getsize(OUTPUT_PATH) / 1e6:.2f} MB)")

try:
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-print_format", "json",
         "-show_streams", "-show_format", OUTPUT_PATH],
        capture_output=True, text=True, check=True).stdout
    info = json.loads(out)
    print("duration:", float(info["format"]["duration"]), "s")
    for s in info["streams"]:
        if s["codec_type"] == "video":
            print(f"video: {s['codec_name']} {s['width']}x{s['height']} "
                  f"@ {s['r_frame_rate']} fps, {s.get('nb_frames', '?')} frames")
        elif s["codec_type"] == "audio":
            print(f"audio: {s['codec_name']} {s['sample_rate']}Hz {s['channels']}ch")
except Exception as e:
    print("ffprobe unavailable:", e)

## 8. Watch the result

In [ ]:
import base64
from IPython.display import HTML, display


def show_video(path, width=512):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(
        f'<video width="{width}" controls autoplay loop playsinline>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'
    ))


show_video(OUTPUT_PATH)

### Confirm numerically that the head does not move

A check straight on the rendered video: crop the forehead/scalp band (well
above the mouth, so lip motion cannot contaminate it) and measure how much it
changes from frame to frame. With the head frozen that is essentially just
encoder noise, while the mouth band moves a lot.

In [ ]:
import numpy as np
import imageio.v3 as iio
import matplotlib.pyplot as plt

frames = np.asarray(iio.imread(OUTPUT_PATH, plugin="pyav"), dtype=np.float32)
n, h, w = frames.shape[:3]

head  = frames[:, :int(h * 0.25)].mean(axis=-1)                 # scalp / forehead
mouth = frames[:, int(h * 0.55):int(h * 0.85)].mean(axis=-1)    # mouth region

d_head  = np.abs(np.diff(head,  axis=0)).mean(axis=(1, 2))
d_mouth = np.abs(np.diff(mouth, axis=0)).mean(axis=(1, 2))

print(f"frames: {n}   size: {w}x{h}")
print(f"mean |delta| head band  : {d_head.mean():.3f}   (max {d_head.max():.3f})")
print(f"mean |delta| mouth band : {d_mouth.mean():.3f}   (max {d_mouth.max():.3f})")
print(f"ratio mouth / head      : {d_mouth.mean() / max(d_head.mean(), 1e-6):.1f}x")

plt.figure(figsize=(11, 3))
plt.plot(d_head,  label="head band (frozen)")
plt.plot(d_mouth, label="mouth band (audio-driven)")
plt.xlabel("frame"); plt.ylabel("mean abs. pixel change"); plt.legend()
plt.title("per-frame change: the head is static, the mouth is not")
plt.tight_layout(); plt.show()

## 9. Download the video

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except ImportError:
    print("not on Colab - the file is at:", OUTPUT_PATH)

## 10. Optional - render the original, free-moving head for comparison

Same inputs, same seed, `freeze_head=False`. Useful as an A/B reference.

In [ ]:
RUN_COMPARISON = False   # flip to True to also render the unfrozen baseline

if RUN_COMPARISON:
    import os, time
    from stream_pipeline_offline import StreamSDK
    from inference import run, seed_everything

    FREE_PATH = os.path.join(os.getcwd(), "tmp", "result_free_head.mp4")
    SDK2 = StreamSDK(CFG_PKL, DATA_ROOT)
    seed_everything(1024)
    t0 = time.time()
    run(SDK2, AUDIO_PATH, SOURCE_PATH, FREE_PATH, freeze_head=False)
    print(f"finished in {time.time() - t0:.1f}s")
    show_video(FREE_PATH)
else:
    print("skipped (set RUN_COMPARISON = True)")

## Notes

### Turning the freeze off

It is on by default everywhere. Three ways to disable it:

```shell
python inference.py \
    --data_root   "./checkpoints/ditto_pytorch" \
    --cfg_pkl     "./checkpoints/ditto_cfg/v0.4_hubert_cfg_pytorch.pkl" \
    --audio_path  "./example/audio.wav" \
    --source_path "./example/image.jpeg" \
    --output_path "./tmp/result.mp4" \
    --no_freeze_head
```

```python
run(SDK, audio_path, source_path, output_path, freeze_head=False)
# or
SDK.setup(source_path, output_path, freeze_head=False)
```

### What changed in this fork

| file | change |
|---|---|
| `core/atomic_components/motion_stitch.py` | `HEAD_POSE_KEYS` / `HEAD_CTRL_KEYS`, the `drop_head_keys` / `drop_head_ctrl_kwargs` / `freeze_head_pose` helpers, and the `freeze_head` flag on `MotionStitch.setup` |
| `stream_pipeline_offline.py`, `stream_pipeline_online.py` | accept and forward `freeze_head` (default `True`) |
| `inference.py` | `freeze_head` argument on `run()` plus the `--no_freeze_head` CLI flag |
| `core/utils/blend/__init__.py` | NumPy fallback when the Cython blend extension cannot be compiled (identical output, just slower) |

Nothing else was touched: the diffusion model, the appearance extractor, the
warping network, the decoder, the eye-blink scheduler, the stitching network
and the put-back/blending stage are all unchanged.

### Troubleshooting

**`OSError: libEGL.so.1: cannot open shared object file`**
This is `mediapipe` failing to load its native library because the Colab image
has no EGL/OpenGL runtime. Fatal - nothing downstream works. Run **cell 3a**,
which `apt-get install`s `libegl1`, `libgl1`, `libglib2.0-0` and friends. If
`mediapipe` was already imported in this kernel before the libs appeared, the
failed import is cached: use **Runtime > Restart session** and rerun from 3a.

**`Failed to load library libonnxruntime_providers_cuda.so ... libcublasLt.so.13`**
Not fatal. The `onnxruntime-gpu` wheel on PyPI is built for CUDA 13, Colab's
torch for CUDA 12, so the GPU provider cannot load and onnxruntime falls back
to CPU - output is identical, only HuBERT feature extraction gets slower. Cell
**3c** fixes it by installing `onnxruntime-gpu==1.22.0` (a CUDA-12 build) and
preloading the CUDA libraries torch bundles. After it reinstalls onnxruntime,
restart the session if the provider check still shows CPU only.

**`Using deprecated NumPy API` / `#warning` from the Cython build**
Harmless compiler noise while `pyximport` builds the blend extension. If the
build fails outright, the NumPy fallback in `core/utils/blend/__init__.py`
takes over and you get a `UserWarning` instead - output is unchanged.

**Other**

* **Out of memory** - use a smaller source image, or lower `max_size`:
  `SDK.setup(..., max_size=1024)`.
* **`ffmpeg: not found` at the mux step** - rerun the check cell at the end of
  section 3; it copies the `imageio-ffmpeg` binary onto `PATH`.
* **A different image/audio** - just point `AUDIO_PATH` / `SOURCE_PATH` in
  section 6 at your own files (`.wav` and an image or video) and rerun from
  section 7.